<center>
    <p style="text-align:center">
    <img alt="arize logo" src="https://storage.googleapis.com/arize-assets/arize-logo-white.jpg" width="300"/>
        <br>
        <a href="https://docs.arize.com/arize/">Docs</a>
        |
        <a href="https://github.com/Arize-ai/client_python">GitHub</a>
        |
        <a href="https://arize-ai.slack.com/join/shared_invite/zt-11t1vbu4x-xkBIHmOREQnYnYDH1GDfCg">Community</a>
    </p>
</center>

# <center>OpenAI Realtime API Voice Tracing</center>

This notebook shows how to trace an OpenAI Realtime voice agent end-to-end with Arize AX, using the [OpenAI Agents SDK](https://github.com/openai/openai-agents-python) and [`openinference-instrumentation-openai-agents`](https://pypi.org/project/openinference-instrumentation-openai-agents/). You will learn:

- How a single `OpenAIAgentsInstrumentor().instrument(...)` call auto-traces an `agents.realtime.RealtimeSession` — capturing audio, transcripts, token counts, and tool calls without any manual span code
- The span tree the instrumentor produces (one `AUDIO` parent per conversational turn, with `USER`, `LLM`, and `TOOL` children)
- The Agents-SDK event stream — `RealtimeAudio`, `RealtimeAudioInterrupted`, `RealtimeAgentEndEvent`, `RealtimeToolStart` — and how to drive a live microphone + speaker loop on top of it
- How to evaluate the captured audio with an audio-aware OpenAI model and log the results back to Arize

Each section runs a small piece of code, then guides you through what to look for in the Arize AX trace view.

### **NOTE**: This notebook must be run locally

Audio capture requires microphone access, which Colab cannot provide.

## What you'll see in Arize

The OpenAI Agents instrumentor turns each turn of a Realtime conversation into a complete span tree, automatically. You write no tracing code yourself — the instrumentor patches `RealtimeSession` and emits spans as audio, tool calls, and responses flow through the WebSocket.

For each conversational turn, the tree looks like this:

```
AUDIO  "conversation.turn"     ← parent; aggregated transcripts, llm.model_name, llm.invocation_parameters
├─ USER "user"                 ← input.audio.url, input.audio.transcript
├─ LLM  "assistant"            ← output.audio.url, output.audio.transcript, token counts, time_to_first_token_ms
│  └─ TOOL "<tool_name>"        ← one per function call within the turn
└─ ...                          ← additional siblings for split input or tool round-trips
```

A few details worth knowing:

- The parent span uses the `AUDIO` [OpenInference span kind](https://github.com/Arize-ai/openinference/tree/main/spec) — Arize AX renders it with audio-aware UI: a play button, waveform, and the input/output transcripts inline.
- `input.audio.url` and `output.audio.url` carry the captured audio as inline `data:audio/wav;base64,…` URIs by default — the bytes ride on the span itself, so no separate audio storage is needed.
- `time_to_first_token_ms` on the assistant span is the latency from the user's audio commit to the first byte of the model's audio response — the metric that matters most for perceived voice-agent responsiveness.
- Token counts include the audio-specific breakdowns `llm.token_count.prompt_details.audio` and `llm.token_count.completion_details.audio`, so you can see audio vs text token cost separately.

## Initial setup

You will need an Arize AX account to run this notebook. [Sign up now for free](https://app.arize.com/auth/join) if you don't have one. You also need an OpenAI API key with access to the Realtime API.

### Install libraries

The cell below installs the OpenAI Agents SDK, the OpenInference instrumentor, the `arize-otel` helper for tracer setup, the `arize` client (used by the evaluation step to export traces and log evaluations back), and `sounddevice` + `numpy` + `pandas` for audio I/O and dataframe work. On Linux it also installs the PortAudio system library that `sounddevice` links against; macOS and Windows wheels bundle PortAudio already.

In [ ]:
import shutil
import sys

if 'google.colab' in sys.modules:
    print("THIS NOTEBOOK MUST RUN LOCALLY. Colab environments do not support microphone capture.")
else:
    # sounddevice's prebuilt wheels bundle PortAudio on macOS and Windows.
    # On Linux you need the system library — install it via the available package manager.
    if sys.platform == "darwin" and shutil.which("brew"):
        !brew list portaudio >/dev/null 2>&1 || brew install portaudio
    elif sys.platform.startswith("linux"):
        if shutil.which("apt-get"):
            !sudo apt-get update && sudo apt-get install -y libportaudio2
        elif shutil.which("dnf"):
            !sudo dnf install -y portaudio
        elif shutil.which("pacman"):
            !sudo pacman -S --noconfirm portaudio
        else:
            print("Install the PortAudio system library before continuing (e.g. apt-get install libportaudio2).")
    # Windows: nothing to do — PortAudio ships in the sounddevice wheel.
    %pip install -q "openai-agents" "openinference-instrumentation-openai-agents" "arize-otel" "arize" "openai" "sounddevice" "numpy" "pandas"

### Imports

We import the standard library pieces (`asyncio`, `os`, `time`, `datetime`, `getpass`), the audio stack (`numpy` and `sounddevice`), and the Agents SDK objects that build the realtime agent. From `agents.realtime.events` we pull the specific event types we'll dispatch on inside the run loop. Finally, `OpenAIAgentsInstrumentor` is the one-line auto-instrumentor that wires the Agents SDK to OpenTelemetry.

In [ ]:
import asyncio
import os
import time
from datetime import datetime
from getpass import getpass

import numpy as np
import sounddevice as sd

from agents import function_tool
from agents.realtime import RealtimeAgent, RealtimeRunner
from agents.realtime.events import (
    RealtimeAgentEndEvent,
    RealtimeAgentStartEvent,
    RealtimeAudio,
    RealtimeAudioInterrupted,
    RealtimeToolStart,
)

from openinference.instrumentation.openai_agents import OpenAIAgentsInstrumentor

### Set up credentials

Enter your OpenAI API key and Arize Space ID + API key. These prompts skip themselves if the env vars are already set.

You can find your Arize Space ID and API key in your space settings: [Arize AX → Space Settings](https://app.arize.com/). `PROJECT_NAME` is the name that will appear in the Arize project picker — change it if you want to group runs separately.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

if not os.environ.get("ARIZE_SPACE_ID"):
    os.environ["ARIZE_SPACE_ID"] = getpass("Enter ARIZE_SPACE_ID: ")

if not os.environ.get("ARIZE_API_KEY"):
    os.environ["ARIZE_API_KEY"] = getpass("Enter ARIZE_API_KEY: ")

PROJECT_NAME = "openai-realtime-voice"

## Defining the voice agent

Three pieces make up the agent we'll wire to the Realtime API: the **tools** (Python functions the model can call), the **agent definition** (instructions and a tool list), and the **audio plumbing** that pipes the microphone in and the speaker out. None of this code is tracing-specific — it's the same shape you'd write without observability. The instrumentor we configure in the next section will trace it all automatically.

### Tools

Two tools: `get_weather` and `get_current_time`. They're trivial dummies (the weather doesn't really come from anywhere), enough to exercise a tool round-trip in the Realtime API.

The `@function_tool` decorator from the Agents SDK introspects the function's type hints and docstring to generate the JSON schema the model receives. You don't write a tool-spec dict by hand. When the model decides to call `get_weather`, the SDK deserialises the arguments, invokes the Python function, serialises the result, and sends it back to the model — all of which the OpenInference instrumentor captures as a child `TOOL` span on the trace.

In [ ]:
@function_tool
def get_weather(location: str, unit: str = "fahrenheit") -> str:
    """Get the current weather for a location."""
    if unit == "celsius":
        return f"The weather in {location} is 22 °C and sunny."
    return f"The weather in {location} is 72 °F and sunny."


@function_tool
def get_current_time(timezone: str = "UTC") -> str:
    """Get the current time for a timezone."""
    now = datetime.now().strftime("%I:%M %p")
    return f"The current time in {timezone} is {now}."

### The agent

A `RealtimeAgent` is the Agents-SDK object that pairs a system prompt with a tool list for the OpenAI Realtime API. It looks identical to a regular `Agent` but is consumed by `RealtimeRunner` instead of `Runner`, which negotiates the WebSocket connection and handles the bidirectional audio stream.

The `instructions` you pass become the session's system message — kept short and explicit about when to call tools and how to reply, since long instructions slow down the time-to-first-token at the start of every turn. You can pass `model="..."` if you want a non-default Realtime model; left unset, the SDK uses its current default.

In [ ]:
agent = RealtimeAgent(
    name="Assistant",
    instructions=(
        "You are a helpful voice assistant. "
        "You have tools to look up weather and the current time — use them when asked. "
        "Keep responses short and conversational."
    ),
    tools=[get_weather, get_current_time],
)

### Audio plumbing

The OpenAI Realtime API speaks 24 kHz PCM16 mono on the wire — that's what `session.send_audio(...)` expects and what `RealtimeAudio` events carry on the way back. We use [`sounddevice`](https://python-sounddevice.readthedocs.io/) to bridge the system's microphone and speaker to that format.

`sounddevice` runs the input stream's callback on a background thread; the callback enqueues mic bytes into an `asyncio.Queue` that the async send loop drains. For playback we use a non-callback `OutputStream` and write to it synchronously via `loop.run_in_executor(...)`, letting PortAudio own the buffering. When the assistant's reply ends, we call `out_stream.stop()`, which blocks until PortAudio has played out the queued audio — so we never need a manual drain loop.

The mic callback also accepts an optional `recording` list. When supplied, every chunk is appended to it with a `time.time_ns()` timestamp. We use this local copy in the [evaluation step](#evaluating-the-voice-session) at the end of the notebook — see that section for the reason it's needed and the conditions under which you can skip it.

In [ ]:
def make_mic_callback(mic_queue, loop, recording=None):
    """Build a sounddevice InputStream callback that forwards mic chunks into an asyncio queue.

    sounddevice invokes the callback on PortAudio's background thread, so we use
    `loop.call_soon_threadsafe` to hand the bytes to the asyncio event loop safely.

    If `recording` is supplied, each chunk is also appended to it as a
    `(time.time_ns(), bytes)` tuple. We use this later to slice out per-USER-span
    audio for evaluation, since Arize stores the inline audio in an internal
    blob bucket we can't fetch from externally.
    """
    def cb(indata, frames, _time, status):
        if status:
            # Underruns, overflows etc. — print but don't crash; PortAudio recovers.
            print(f"Mic: {status}")
        # `indata` is a view onto a buffer sounddevice reuses on the next callback,
        # so we copy before handing the bytes to a different thread.
        chunk = indata.copy().tobytes()
        if recording is not None:
            recording.append((time.time_ns(), chunk))
        loop.call_soon_threadsafe(mic_queue.put_nowait, chunk)
    return cb

## Setting up tracing

This is the only tracing code you need to write for the whole notebook — one `register(...)` call and one `OpenAIAgentsInstrumentor().instrument(...)` call. The instrumentor patches `agents.realtime.RealtimeSession` to emit the span tree shown above; all spans flow through the registered `TracerProvider` to Arize AX.

By default, captured audio rides inline on the span attributes as `data:audio/wav;base64,…` URIs — no separate storage needed. The inline payload is capped by `OPENINFERENCE_BASE64_AUDIO_MAX_LENGTH` (default `32000` characters ≈ 0.5 s of 24 kHz mono PCM16). We raise it to `2000000` (~30 s) below, which is plenty for typical voice turns. **Set this env var before calling `instrument(...)`** — the instrumentor reads it at patch time.

<Tip>
If you need durable storage or longer turns than the inline cap allows, swap the audio URL for any HTTPS endpoint Arize AX can fetch — an S3 presigned URL, a GCS object, your own CDN. Attach a custom `SpanProcessor` or wrap the `SpanExporter` to substitute `input.audio.url` / `output.audio.url` with your chosen URL before the span leaves the process. Arize plays back whatever URL the attribute carries.

Using an externally fetchable URL also **removes the need for the local-recording workaround** in the evaluation step below — see the [eval section](#evaluating-the-voice-session) for the details.
</Tip>

In [ ]:
from arize.otel import register

# Raise the inline base64 cap to ~30 s of audio (default is ~0.5 s).
# Must be set BEFORE .instrument() — the instrumentor reads it at patch time.
os.environ["OPENINFERENCE_BASE64_AUDIO_MAX_LENGTH"] = "2000000"

# arize.otel.register builds a TracerProvider with an OTLP exporter wired to Arize AX,
# and installs it as the global OpenTelemetry tracer provider.
tracer_provider = register(
    space_id=os.environ["ARIZE_SPACE_ID"],
    api_key=os.environ["ARIZE_API_KEY"],
    project_name=PROJECT_NAME,
)

# Patch agents.realtime.RealtimeSession so it emits spans through our tracer provider.
OpenAIAgentsInstrumentor().instrument(tracer_provider=tracer_provider)

## Running a voice session

This is the live mic/speaker loop. It opens an `agents.realtime.RealtimeSession` via the `RealtimeRunner`, pumps mic audio into it, plays back the assistant's audio, and exits cleanly after one full conversational turn — including any tool calls and the assistant's follow-up response that uses the tool result.

A few things worth understanding before running it:

**Three async tasks run concurrently inside the session:**

- `send_mic` — drains the mic queue and forwards chunks to the session via `session.send_audio(...)`
- `handle_events` — consumes the session's event stream (`async for event in session`) and dispatches on type:
  - `RealtimeAudio` → write to the output stream (queued for playback)
  - `RealtimeAudioInterrupted` → user spoke over the assistant; abort and re-arm the output stream
  - `RealtimeAgentStartEvent` / `RealtimeToolStart` → "more is coming" — cancel any pending exit
  - `RealtimeAgentEndEvent` → "a response just ended" — schedule a deferred exit
- `hard_timer` — sets `stop` after `MAX_SESSION_SECONDS` as a safety net

**Why the exit is deferred** rather than fired directly on `RealtimeAgentEndEvent`: the SDK's `RealtimeAgentEndEvent` fires after every `response.done` from the API, which includes the response where the model decides to call a tool — *before* the tool runs and the follow-up response. So we schedule the exit after `EXIT_GRACE_PERIOD`, and cancel it if a `RealtimeToolStart` or a new `RealtimeAgentStartEvent` follows within the window (both indicate the turn isn't really over). For tool-free turns the grace window simply elapses and the session exits.

**Try asking:**

- *"What's the weather in London?"* — exercises a tool call (`get_weather`)
- *"What time is it in Tokyo?"* — exercises the other tool (`get_current_time`)
- *"Tell me a fun fact about Python"* — no tool call; tests the simpler one-response path

Server-side VAD (the Realtime API's default `turn_detection`) detects when you've finished speaking and commits your audio buffer automatically. Interrupt the kernel to stop early.

In [ ]:
SAMPLE_RATE = 24_000             # OpenAI Realtime API speaks 24 kHz PCM16 mono on the wire
CHANNELS = 1
MIC_CHUNK_FRAMES = 1_200         # 50 ms blocks
MAX_SESSION_SECONDS = 60         # hard ceiling — session ends after this no matter what
EXIT_GRACE_PERIOD = 1.5          # seconds after AgentEndEvent before exit fires, unless cancelled

# Captured mic chunks with ns-precision timestamps. Populated by the mic callback
# during the session, then sliced by the eval step to extract per-USER-span audio.
mic_recording: list[tuple[int, bytes]] = []


async def run_session():
    loop = asyncio.get_running_loop()

    # mic_queue: PortAudio thread -> async send loop.
    # stop: set when we want every task (mic, events, timer) to wind down.
    mic_queue: asyncio.Queue = asyncio.Queue(maxsize=100)
    stop = asyncio.Event()

    # Start each session with a fresh recording.
    mic_recording.clear()

    async def hard_timer():
        """Safety net — guarantees the session exits even if no events fire."""
        await asyncio.sleep(MAX_SESSION_SECONDS)
        stop.set()

    runner = RealtimeRunner(agent)
    # `runner.run()` returns an async context manager that opens the Realtime
    # WebSocket session; `await` it once, then enter with `async with`.
    async with await runner.run() as session:
        print("Connected. Speak now — session will end once the assistant finishes its turn.\n")

        async def send_mic():
            """Drain mic_queue and forward chunks to the session.

            `wait_for(..., timeout=0.1)` lets us check `stop` ~10x/s instead of
            blocking forever on an empty queue.
            """
            while not stop.is_set():
                try:
                    chunk = await asyncio.wait_for(mic_queue.get(), timeout=0.1)
                    await session.send_audio(chunk)
                except asyncio.TimeoutError:
                    continue

        timer_task = asyncio.create_task(hard_timer())
        send_task = asyncio.create_task(send_mic())
        events_task = None  # created below once the audio streams are open

        try:
            # Open mic + speaker streams. They start automatically on `with` entry
            # and are closed on exit. The output stream uses no callback — we'll
            # write to it synchronously from handle_events.
            with sd.InputStream(
                samplerate=SAMPLE_RATE,
                channels=CHANNELS,
                dtype="int16",
                blocksize=MIC_CHUNK_FRAMES,
                callback=make_mic_callback(mic_queue, loop, recording=mic_recording),
            ), sd.OutputStream(
                samplerate=SAMPLE_RATE,
                channels=CHANNELS,
                dtype="int16",
                blocksize=MIC_CHUNK_FRAMES,
            ) as out_stream:

                async def handle_events():
                    """Consume the session's event stream and drive playback + exit logic."""
                    exit_task = None  # the pending "session done" timer, if any

                    async def maybe_exit():
                        """Wait the grace period, then drain playback and stop the session."""
                        try:
                            await asyncio.sleep(EXIT_GRACE_PERIOD)
                            # `out_stream.stop()` is blocking — it returns only after
                            # PortAudio has finished playing every queued sample.
                            await loop.run_in_executor(None, out_stream.stop)
                            stop.set()
                        except asyncio.CancelledError:
                            pass

                    def cancel_pending_exit():
                        nonlocal exit_task
                        if exit_task is not None and not exit_task.done():
                            exit_task.cancel()
                        exit_task = None

                    async for event in session:
                        if stop.is_set():
                            break
                        if isinstance(event, RealtimeAudio):
                            # Assistant audio chunk — write to the speaker. run_in_executor
                            # keeps the blocking PortAudio write off the event loop.
                            arr = np.frombuffer(event.audio.data, dtype=np.int16)
                            await loop.run_in_executor(None, out_stream.write, arr)
                        elif isinstance(event, RealtimeAudioInterrupted):
                            # User talked over the assistant — drop queued audio and
                            # re-arm the stream for the next reply.
                            cancel_pending_exit()
                            await loop.run_in_executor(None, out_stream.abort)
                            await loop.run_in_executor(None, out_stream.start)
                            print("[interrupted]")
                        elif isinstance(event, (RealtimeAgentStartEvent, RealtimeToolStart)):
                            # New response or tool call — there's more coming.
                            cancel_pending_exit()
                        elif isinstance(event, RealtimeAgentEndEvent):
                            # A response just ended. Maybe final, maybe before a tool call.
                            # Schedule exit; a follow-up AgentStart or ToolStart will cancel it.
                            cancel_pending_exit()
                            exit_task = asyncio.create_task(maybe_exit())

                events_task = asyncio.create_task(handle_events())
                await stop.wait()
        except KeyboardInterrupt:
            print("\nInterrupted.")
            stop.set()
        finally:
            # Cancel any task that was actually created and wait for them to unwind.
            pending = [t for t in (timer_task, send_task, events_task) if t is not None]
            for t in pending:
                t.cancel()
            await asyncio.gather(*pending, return_exceptions=True)

    # Ensure every batched span reaches Arize before the cell returns.
    tracer_provider.force_flush()
    print(f"Session ended. Captured {len(mic_recording)} mic chunks. Traces flushed to Arize.")


await run_session()

## See your traces in Arize

Head to your Arize AX project (`openai-realtime-voice`, unless you changed `PROJECT_NAME`) to see the traces. Each conversation turn appears as a `conversation.turn` span (kind: `AUDIO`) containing `user`, `assistant`, and any `<tool_name>` children.

![An audio trace](https://storage.googleapis.com/arize-phoenix-assets/assets/images/arize-docs-images/cookbooks/voice/trace-audio.png)

Things to look at in the trace view:

- **Audio playback** — click the play button on the `user` and `assistant` spans. The audio is served from the inline `data:audio/wav;base64,...` URI on the span attribute.
- **Transcripts** — the `input.audio.transcript` and `output.audio.transcript` attributes show what the model heard and what it said. Useful for debugging mishearings.
- **Tool calls** — child `TOOL` spans under the `assistant` span show the function name, arguments JSON, and the value the function returned.
- **Latency** — `time_to_first_token_ms` on the `assistant` span gives the user-perceived "how fast did the agent start talking" latency. Look for outliers across turns.
- **Audio token cost** — `llm.token_count.prompt_details.audio` and `llm.token_count.completion_details.audio` break out audio tokens separately from text tokens.

### Audio redaction

The OpenInference instrumentor recognises three environment variables for controlling what audio data ends up on spans. Set them **before** calling `instrument(...)`:

- `OPENINFERENCE_HIDE_INPUT_AUDIO=true` — drop `input.audio.*` from `USER` spans
- `OPENINFERENCE_HIDE_OUTPUT_AUDIO=true` — drop `output.audio.*` from `LLM` spans
- `OPENINFERENCE_BASE64_AUDIO_MAX_LENGTH=<n>` — cap the inline base64 payload length (default `32000`)

The general OpenInference `TraceConfig(hide_inputs=True)` and `TraceConfig(hide_outputs=True)` settings also cascade to the corresponding audio attributes, so a single redaction config covers both text and audio.

## Evaluating the voice session

Now that traces are flowing into Arize, let's run an evaluation against the captured audio. We'll classify the **tone of each user utterance** as `positive`, `neutral`, or `negative` — a classic voice-agent quality signal — and ship the eval results back to Arize so they appear on the same spans in the UI.

There's one architectural wrinkle to know about up front. When the OpenInference instrumentor uploads inline `data:audio/wav;base64,…` URIs to Arize, the backend stores the audio in its own multimodal blob bucket and the export client returns an internal `gs://arize-multimodal-prod/…` reference that external code can't authenticate against. We work around this by **keeping a local copy of the mic audio** as it's captured (the `mic_recording` list populated by the mic callback), and slicing it per USER span using the span's start and end times from the Arize export.

<Note>
**If you swap the inline-base64 path for external cloud storage** (the [tip in the tracing section](#setting-up-tracing) shows how — a custom `SpanProcessor` that puts an S3 / GCS / CDN URL on `input.audio.url` instead of the inline data URI), **this workaround disappears**. The audio URL on each USER span points at an object you control and can fetch directly. You can then drop the `recording=mic_recording` argument from `make_mic_callback`, delete the `extract_audio_b64` slicing helper, and replace it with a plain HTTP fetch of `attributes.input.audio.url`. The local-recording dance below exists only because the default inline-base64 path stores audio in Arize-controlled storage.
</Note>

The flow has four steps:

1. **Export** the USER spans from Arize — we only need them for span IDs and time bounds, not the audio bytes
2. **Slice** `mic_recording` between each USER span's `start_time` and `end_time`, wrap the raw PCM in a WAV header
3. **Classify** the resulting audio with `gpt-audio-mini` (OpenAI's audio-input chat model)
4. **Log** the evaluations back to Arize via `client.spans.update_evaluations(...)`

<Note>
Phoenix's `phoenix.evals` framework was rewritten in late 2025 and its prompt templates currently support **text content only** — multimodal/audio content blocks aren't supported yet. So instead of using `create_classifier(...)`, we call OpenAI directly and shape the result into the eval-dataframe format Arize expects.
</Note>

In [ ]:
import base64
import io
import json
import wave
from datetime import datetime, timedelta, timezone

import pandas as pd
from arize import ArizeClient
from openai import OpenAI

# Clients for the two halves of the eval round-trip.
arize_client = ArizeClient(api_key=os.environ["ARIZE_API_KEY"])
openai_client = OpenAI()

### Export the traces from Arize

`client.spans.export_to_df(...)` returns a flat pandas `DataFrame` with one row per span. We pull the last hour of spans and filter to `USER` spans — these are the per-turn user-input spans that the instrumentor creates when the server-side VAD detects the end of an utterance.

We only need three columns from the export: `context.span_id` (where the eval attaches), `start_time`, and `end_time`. The audio bytes themselves come from `mic_recording`.

In [ ]:
# 1-hour lookback covers the session you just ran plus a generous margin for
# Arize's ingestion latency. Narrow the window if you have a noisy project.
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=1)

traces_df = arize_client.spans.export_to_df(
    space_id=os.environ["ARIZE_SPACE_ID"],
    project_name=PROJECT_NAME,
    start_time=start_time,
    end_time=end_time,
)
print(f"Exported {len(traces_df)} spans")

# USER spans are the per-utterance children of each AUDIO turn. Their
# start_time / end_time bound the audio we want to evaluate.
user_spans = traces_df[
    traces_df.get("attributes.openinference.span.kind") == "USER"
].copy()
print(f"{len(user_spans)} user spans to evaluate")

### Slice the user audio out of the local recording

For each USER span, slice `mic_recording` between the span's `start_time` and `end_time` (both `pandas.Timestamp` values, convertible to ns-since-epoch via `.value`). The slice is raw PCM16 mono at 24 kHz; we wrap it in a WAV header using the standard-library `wave` module, base64-encode the result, and hand it to the classifier.

If no mic chunk falls in the span's window, the helper returns `None` and we skip that span — that can happen for very short utterances right at the start or end of the recording.

In [ ]:
def _ts_to_ns(ts) -> int:
    """Convert a pandas/datetime/string timestamp to ns since the Unix epoch."""
    return int(pd.Timestamp(ts).value)


def _wrap_pcm_as_wav_b64(pcm_bytes: bytes) -> str:
    """Wrap raw PCM16 mono 24 kHz bytes in a WAV header and return base64."""
    buf = io.BytesIO()
    with wave.open(buf, "wb") as wav:
        wav.setnchannels(CHANNELS)
        wav.setsampwidth(2)  # int16
        wav.setframerate(SAMPLE_RATE)
        wav.writeframes(pcm_bytes)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def extract_audio_b64(span, recording=mic_recording) -> str | None:
    """Slice the recording by the span's [start_time, end_time] and return WAV base64."""
    start_ns = _ts_to_ns(span["start_time"])
    end_ns = _ts_to_ns(span["end_time"])
    pcm = b"".join(chunk for ts, chunk in recording if start_ns <= ts <= end_ns)
    if not pcm:
        return None
    return _wrap_pcm_as_wav_b64(pcm)

### Define the tone classifier

The classifier mirrors what a classic emotion-classification template does, just spelled out against the OpenAI SDK directly:

1. **Task description** — a system prompt that tells the model what to listen for and how to reply
2. **Rails** — the fixed set of valid labels (`positive` / `neutral` / `negative`); the prompt forbids anything else
3. **Output format** — a small JSON object with `label` and `explanation`, parsed back into the score columns Arize expects

`gpt-audio-mini` accepts an `input_audio` content block alongside text, so we send the system prompt as one message and a short text + audio block as the user message. `modalities=["text"]` tells the model to reply in text only (not audio). The full `gpt-audio` model would also work — `gpt-audio-mini` is cheaper and plenty accurate for three-way tone classification.

<Note>
Audio chat models don't support `response_format={"type": "json_object"}` (yet). We pin the output shape with a strong prompt instruction and parse leniently — `_extract_json_object` pulls the first `{...}` block out of the response, which survives minor deviations like the model wrapping its reply in markdown fences.
</Note>

In [ ]:
import re

# Three-way tone classification. The model is constrained to these labels by
# the prompt; the score mapping turns each label into a numeric value Arize
# can sort and chart on (any monotonic mapping works — 0 / 0.5 / 1 is just
# convenient for filtering "tone went negative" in the trace list).
TONE_RAILS = ["positive", "neutral", "negative"]
TONE_SCORES = {"positive": 1.0, "neutral": 0.5, "negative": 0.0}

TONE_SYSTEM_PROMPT = (
    "You are an evaluator that listens to a user's voice clip and classifies the tone of voice. "
    f"Reply with one of exactly these labels: {', '.join(TONE_RAILS)}. "
    "Respond with ONLY a JSON object on a single line, with two keys: "
    '"label" (one of the labels above) and "explanation" (a one-sentence reason). '
    "Do not wrap the JSON in markdown, do not include any preamble, do not add any other text."
)


def _extract_json_object(text: str) -> dict:
    """Pull the first {...} object out of the model's response and parse it.

    Audio models don't support `response_format={"type": "json_object"}`, so we
    rely on prompt instructions and a lenient extractor that survives minor
    deviations like the model wrapping its reply in markdown fences.
    """
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON object in classifier response: {text!r}")
    return json.loads(match.group(0))


def classify_tone(audio_b64: str) -> dict:
    """Classify the tone of one user utterance. Returns {label, score, explanation}."""
    response = openai_client.chat.completions.create(
        model="gpt-audio-mini",
        # We want text out (the JSON label), not synthesized audio.
        modalities=["text"],
        messages=[
            {"role": "system", "content": TONE_SYSTEM_PROMPT},
            {"role": "user", "content": [
                {"type": "text", "text": "Classify the tone of this audio clip:"},
                # `input_audio` content blocks accept base64-encoded WAV directly.
                {"type": "input_audio", "input_audio": {"data": audio_b64, "format": "wav"}},
            ]},
        ],
    )
    parsed = _extract_json_object(response.choices[0].message.content)
    label = parsed.get("label", "neutral").lower()
    if label not in TONE_RAILS:
        # Model went off-rails — snap to neutral so the eval row still lands.
        label = "neutral"
    return {
        "label": label,
        "score": TONE_SCORES[label],
        "explanation": parsed.get("explanation", ""),
    }

### Run the evaluation and log results back to Arize

Loop over each user-audio span, classify, and assemble an eval dataframe in the exact shape `update_evaluations` requires:

- `context.span_id` — the span the eval attaches to
- `eval.<name>.label` — string label (one of the rails)
- `eval.<name>.score` — numeric score (0.0 / 0.5 / 1.0 for negative / neutral / positive)
- `eval.<name>.explanation` — free-text justification from the model

Once it's logged, the eval appears on each USER span in the Arize trace view alongside the audio playback and transcript.

In [ ]:
# Build one row per USER span in the shape Arize's `update_evaluations`
# requires: `context.span_id` plus `eval.<name>.{label,score,explanation}` columns.
rows = []
for _, span in user_spans.iterrows():
    audio_b64 = extract_audio_b64(span)
    if audio_b64 is None:
        print(f"  skipping {span['context.span_id']}: no mic audio in span window")
        continue
    result = classify_tone(audio_b64)
    rows.append({
        "context.span_id": span["context.span_id"],
        "eval.tone.label": result["label"],
        "eval.tone.score": result["score"],
        "eval.tone.explanation": result["explanation"],
    })

evals_df = pd.DataFrame(rows)
print(evals_df)

arize_client.spans.update_evaluations(
    space_id=os.environ["ARIZE_SPACE_ID"],
    project_name=PROJECT_NAME,
    dataframe=evals_df,
)
print(f"Logged {len(evals_df)} tone evaluations to Arize.")

### Review the evaluation in Arize

Refresh your Arize AX project. Each USER span in the trace now carries the `tone` evaluation alongside the audio playback and transcript — open a trace and you'll see the label, score, and the model's explanation:

![Audio tone evaluation shown on the USER span in the Arize AX trace view](https://storage.googleapis.com/arize-phoenix-assets/assets/images/arize-docs-images/cookbooks/voice/audio-eval.png)

The label and score are filterable in the trace list view, so you can quickly find sessions where tone went negative — useful for digging into failure modes or sampling for regression review.

## Read more

- [OpenInference OpenAI Agents instrumentor](https://github.com/Arize-ai/openinference/tree/main/python/instrumentation/openinference-instrumentation-openai-agents) — source, changelog, and the canonical `realtime_with_tools.py` example this notebook is adapted from
- [OpenAI Agents SDK realtime docs](https://openai.github.io/openai-agents-python/realtime/quickstart/) — `RealtimeAgent`, `RealtimeRunner`, and the full event reference
- [OpenAI Realtime API reference](https://platform.openai.com/docs/guides/realtime) — the underlying WebSocket protocol the SDK speaks
- [OpenAI audio input guide](https://platform.openai.com/docs/guides/audio) — the `gpt-audio` / `gpt-audio-mini` API used by the evaluator
- [OpenInference semantic conventions](https://github.com/Arize-ai/openinference/tree/main/spec) — the formal definitions for span kinds and audio attributes
- [Arize AX docs — Tracing and evaluating voice applications](https://arize.com/docs/ax/cookbooks/evaluation/tracing-and-evaluating-audio) — the docs companion to this notebook